# 01 - 损失函数（Loss Functions）

## 学习目标

1. 理解损失函数在训练中的作用
2. 掌握回归任务常用的损失函数（L1Loss、MSELoss、SmoothL1Loss）
3. 掌握分类任务常用的损失函数（CrossEntropyLoss、BCELoss、NLLLoss）
4. 了解知识蒸馏（KLDivLoss）和度量学习（TripletMarginLoss）的损失函数
5. 能够根据任务选择合适的损失函数

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## 1. 损失函数概述

损失函数衡量模型预测值与真实值之间的差距，是优化的目标函数。

```
模型输出 (prediction) ──┐
                        ├── 损失函数 ──> loss（标量）──> backward() ──> 更新参数
真实标签 (target) ──────┘
```

PyTorch 中所有损失函数都继承自 `nn.Module`，可以像模型一样使用。

### reduction 参数

大多数损失函数都有 `reduction` 参数，控制输出方式：

| reduction | 行为 | 输出 |
|-----------|------|------|
| `'none'` | 不做任何聚合 | 与输入同形状的张量 |
| `'mean'` | 取平均值（默认） | 标量 |
| `'sum'` | 求和 | 标量 |

## 2. 回归损失函数

### 2.1 L1Loss（平均绝对误差 MAE）

$$L = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

特点：对异常值不敏感，梯度恒定（不为零处）

In [ ]:
# L1Loss 示例
pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.5, 2.5, 3.5])

# 三种 reduction 模式
for mode in ['none', 'mean', 'sum']:
    loss_fn = nn.L1Loss(reduction=mode)
    loss = loss_fn(pred, target)
    print(f"reduction='{mode}': {loss}")

# 手动验证
manual = torch.abs(pred - target)
print(f"\n手动计算:")
print(f"  |pred - target| = {manual}")
print(f"  mean = {manual.mean():.4f}")
print(f"  sum  = {manual.sum():.4f}")

### 2.2 MSELoss（均方误差）

$$L = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$

特点：对异常值敏感，梯度随误差增大而增大

In [ ]:
# MSELoss 示例
pred = torch.tensor([1.0, 2.0, 3.0])
target = torch.tensor([1.5, 2.5, 3.5])

for mode in ['none', 'mean', 'sum']:
    loss_fn = nn.MSELoss(reduction=mode)
    loss = loss_fn(pred, target)
    print(f"reduction='{mode}': {loss}")

# 手动验证
manual = (pred - target) ** 2
print(f"\n手动计算:")
print(f"  (pred - target)^2 = {manual}")
print(f"  mean = {manual.mean():.4f}")

### 2.3 L1Loss vs MSELoss 对比

通过一个含异常值的例子对比两者的差异

In [ ]:
# 对比：异常值的影响
pred_normal = torch.tensor([1.0, 2.0, 3.0, 4.0])
target_normal = torch.tensor([1.1, 2.1, 3.1, 4.1])

# 加入一个异常值
pred_outlier = torch.tensor([1.0, 2.0, 3.0, 4.0])
target_outlier = torch.tensor([1.1, 2.1, 3.1, 14.0])  # 最后一个是异常值

l1 = nn.L1Loss()
mse = nn.MSELoss()

print("正常数据:")
print(f"  L1Loss:  {l1(pred_normal, target_normal):.4f}")
print(f"  MSELoss: {mse(pred_normal, target_normal):.4f}")

print("\n含异常值:")
print(f"  L1Loss:  {l1(pred_outlier, target_outlier):.4f}")
print(f"  MSELoss: {mse(pred_outlier, target_outlier):.4f}")

print("\n结论: MSELoss 对异常值更敏感（平方放大了误差）")

### 2.4 SmoothL1Loss / HuberLoss

结合了 L1Loss 和 MSELoss 的优点：

$$L = \begin{cases} 0.5(y - \hat{y})^2 / \beta & \text{if } |y - \hat{y}| < \beta \\ |y - \hat{y}| - 0.5\beta & \text{otherwise} \end{cases}$$

- 误差小时：使用 MSE（梯度平滑）
- 误差大时：使用 L1（对异常值鲁棒）

In [ ]:
# SmoothL1Loss 示例
pred = torch.tensor([1.0, 2.0, 3.0, 4.0])
target = torch.tensor([1.1, 2.5, 5.0, 14.0])

smooth_l1 = nn.SmoothL1Loss(reduction='none')  # beta 默认为 1.0
huber = nn.HuberLoss(reduction='none', delta=1.0)  # delta 等价于 beta

print(f"误差:       {torch.abs(pred - target)}")
print(f"SmoothL1:   {smooth_l1(pred, target)}")
print(f"HuberLoss:  {huber(pred, target)}")

# 手动验证
diff = torch.abs(pred - target)
beta = 1.0
manual = torch.where(
    diff < beta,
    0.5 * diff ** 2 / beta,  # 小误差：MSE
    diff - 0.5 * beta         # 大误差：L1
)
print(f"手动计算:   {manual}")
print(f"\n说明: SmoothL1Loss 和 HuberLoss 本质相同，参数名不同（beta vs delta）")

## 3. 分类损失函数

### 3.1 CrossEntropyLoss（交叉熵损失）

多分类任务最常用的损失函数。

$$L = -\sum_{c=1}^{C} y_c \log(\hat{y}_c)$$

PyTorch 的 `CrossEntropyLoss` 内部包含了 `Softmax`，所以输入应该是**原始 logits**，不需要先过 Softmax。

```
CrossEntropyLoss = Softmax + Log + NLLLoss
```

In [ ]:
# CrossEntropyLoss 基本用法
# 3 个样本，4 个类别
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],   # 样本 0：预测偏向类别 0
    [0.5, 2.5, 0.3, 0.2],   # 样本 1：预测偏向类别 1
    [0.1, 0.2, 3.0, 0.1],   # 样本 2：预测偏向类别 2
])
targets = torch.tensor([0, 1, 2])  # 真实标签

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)
print(f"CrossEntropyLoss: {loss:.4f}")

In [ ]:
# 手动实现 CrossEntropyLoss = Softmax + Log + NLLLoss
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],
    [0.5, 2.5, 0.3, 0.2],
    [0.1, 0.2, 3.0, 0.1],
])
targets = torch.tensor([0, 1, 2])

# 步骤 1：Softmax
softmax_output = F.softmax(logits, dim=1)
print(f"Softmax 输出:\n{softmax_output}")
print(f"每行之和: {softmax_output.sum(dim=1)}")

# 步骤 2：取 Log
log_softmax_output = torch.log(softmax_output)
print(f"\nLog Softmax 输出:\n{log_softmax_output}")

# 步骤 3：NLLLoss（取出对应类别的负对数概率，再取平均）
nll_values = []
for i, t in enumerate(targets):
    nll_values.append(-log_softmax_output[i, t])
    print(f"样本 {i}: 类别 {t.item()}, -log(p) = {-log_softmax_output[i, t]:.4f}")

manual_loss = sum(nll_values) / len(nll_values)
print(f"\n手动计算的 loss: {manual_loss:.4f}")

# 对比 PyTorch 的结果
pytorch_loss = nn.CrossEntropyLoss()(logits, targets)
print(f"PyTorch 的 loss:  {pytorch_loss:.4f}")
print(f"两者一致: {torch.allclose(manual_loss, pytorch_loss)}")

In [ ]:
# CrossEntropyLoss 的 weight 参数：处理类别不平衡
# 假设 4 个类别，样本数量比例为 100:50:20:10
# 给样本少的类别更高的权重
class_counts = torch.tensor([100.0, 50.0, 20.0, 10.0])
weights = 1.0 / class_counts
weights = weights / weights.sum() * len(weights)  # 归一化
print(f"类别权重: {weights}")

logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],
    [0.5, 2.5, 0.3, 0.2],
    [0.1, 0.2, 3.0, 0.1],
])
targets = torch.tensor([0, 1, 2])

loss_no_weight = nn.CrossEntropyLoss()(logits, targets)
loss_weighted = nn.CrossEntropyLoss(weight=weights)(logits, targets)

print(f"\n无权重 loss: {loss_no_weight:.4f}")
print(f"加权 loss:   {loss_weighted:.4f}")
print("\n说明: 权重让模型更关注少数类")

In [ ]:
# CrossEntropyLoss 的 label_smoothing 参数
# 标签平滑：防止模型过于自信
# 将 one-hot 标签 [1, 0, 0, 0] 变成 [0.925, 0.025, 0.025, 0.025]（smoothing=0.1）

logits = torch.tensor([[2.0, 1.0, 0.1, 0.5]])
target = torch.tensor([0])

for smoothing in [0.0, 0.1, 0.2, 0.3]:
    loss = nn.CrossEntropyLoss(label_smoothing=smoothing)(logits, target)
    print(f"label_smoothing={smoothing}: loss={loss:.4f}")

print("\n说明: label_smoothing 让标签不再是硬标签")
print("  smoothing=0.0: 标签 = [1.0, 0.0, 0.0, 0.0]")
print("  smoothing=0.1: 标签 = [0.925, 0.025, 0.025, 0.025]")
print("  好处: 防止过拟合，提高泛化能力")

### 3.2 NLLLoss（负对数似然损失）

$$L = -\frac{1}{n} \sum_{i=1}^{n} \log(\hat{y}_{i, c_i})$$

注意：NLLLoss 的输入需要是 **log 概率**（即先经过 `LogSoftmax`）

```
CrossEntropyLoss(logits, targets)
= NLLLoss(LogSoftmax(logits), targets)
```

In [ ]:
# 验证: CrossEntropyLoss = LogSoftmax + NLLLoss
logits = torch.tensor([
    [2.0, 1.0, 0.1, 0.5],
    [0.5, 2.5, 0.3, 0.2],
])
targets = torch.tensor([0, 1])

# 方式 1: CrossEntropyLoss
loss_ce = nn.CrossEntropyLoss()(logits, targets)

# 方式 2: LogSoftmax + NLLLoss
log_softmax = nn.LogSoftmax(dim=1)
nll_loss = nn.NLLLoss()
loss_nll = nll_loss(log_softmax(logits), targets)

print(f"CrossEntropyLoss:       {loss_ce:.6f}")
print(f"LogSoftmax + NLLLoss:   {loss_nll:.6f}")
print(f"两者一致: {torch.allclose(loss_ce, loss_nll)}")

print("\n通常直接用 CrossEntropyLoss 即可，除非你需要自定义中间步骤")

### 3.3 BCELoss vs BCEWithLogitsLoss

二分类任务使用的损失函数。

$$L = -\frac{1}{n} \sum [y \cdot \log(\hat{y}) + (1-y) \cdot \log(1-\hat{y})]$$

- `BCELoss`：输入需要先经过 `Sigmoid`（概率值 0~1）
- `BCEWithLogitsLoss`：输入是原始 logits（内部自动做 Sigmoid）

**推荐使用 `BCEWithLogitsLoss`**，原因：
1. 数值更稳定（使用 log-sum-exp 技巧避免数值溢出）
2. 减少一步操作，代码更简洁

In [ ]:
# BCELoss vs BCEWithLogitsLoss
logits = torch.tensor([0.5, -1.0, 2.0, -0.5])
targets = torch.tensor([1.0, 0.0, 1.0, 0.0])

# 方式 1: Sigmoid + BCELoss
probs = torch.sigmoid(logits)
loss_bce = nn.BCELoss()(probs, targets)

# 方式 2: BCEWithLogitsLoss（推荐）
loss_bce_logits = nn.BCEWithLogitsLoss()(logits, targets)

print(f"logits:    {logits}")
print(f"sigmoid:   {probs}")
print(f"targets:   {targets}")
print(f"\nBCELoss:             {loss_bce:.6f}")
print(f"BCEWithLogitsLoss:   {loss_bce_logits:.6f}")
print(f"两者一致: {torch.allclose(loss_bce, loss_bce_logits)}")

In [ ]:
# 为什么 BCEWithLogitsLoss 更稳定？
# 当 logits 很大或很小时，sigmoid 会饱和
extreme_logits = torch.tensor([100.0, -100.0])  # 极端值
targets = torch.tensor([1.0, 0.0])

# Sigmoid + BCELoss：可能出现数值问题
probs = torch.sigmoid(extreme_logits)
print(f"极端 logits: {extreme_logits}")
print(f"sigmoid 后:  {probs}")
print(f"log(sigmoid): {torch.log(probs)}")
# 注意：log(1.0) = 0，但实际 sigmoid(100) ≈ 1.0 有精度损失

# BCEWithLogitsLoss：内部使用更稳定的计算方式
loss = nn.BCEWithLogitsLoss()(extreme_logits, targets)
print(f"\nBCEWithLogitsLoss: {loss:.6f}")
print("\n结论: 始终使用 BCEWithLogitsLoss，让 PyTorch 处理数值稳定性")

## 4. 知识蒸馏：KLDivLoss

KL 散度衡量两个概率分布之间的差异，常用于知识蒸馏（将大模型的知识迁移到小模型）。

$$D_{KL}(P \| Q) = \sum P(x) \cdot \log\frac{P(x)}{Q(x)}$$

PyTorch 中：
- **输入**：学生模型的 **log 概率**（`F.log_softmax`）
- **目标**：教师模型的 **概率**（`F.softmax`）

In [ ]:
# 知识蒸馏示例
# 教师模型（大模型）的 logits
teacher_logits = torch.tensor([[5.0, 3.0, 1.0, 0.5]])
# 学生模型（小模型）的 logits
student_logits = torch.tensor([[4.0, 2.5, 1.5, 1.0]])

# 温度参数 T：T 越大，分布越平滑，传递更多 "暗知识"
T = 4.0

# 计算软标签
teacher_soft = F.softmax(teacher_logits / T, dim=1)
student_log_soft = F.log_softmax(student_logits / T, dim=1)

print(f"教师软标签: {teacher_soft}")
print(f"学生 log 软标签: {student_log_soft}")

# KL 散度损失
kl_loss = nn.KLDivLoss(reduction='batchmean')
loss = kl_loss(student_log_soft, teacher_soft) * (T * T)
print(f"\nKL 散度 loss: {loss:.4f}")

print("\n知识蒸馏总损失 = alpha * CE_loss + (1-alpha) * KL_loss")
print("  CE_loss: 学生 vs 真实标签（硬标签）")
print("  KL_loss: 学生 vs 教师（软标签）")

## 5. 度量学习：TripletMarginLoss

用于学习样本之间的相对距离关系。

$$L = \max(d(a, p) - d(a, n) + \text{margin}, 0)$$

- **anchor**（锚点）：参考样本
- **positive**（正样本）：与 anchor 同类的样本
- **negative**（负样本）：与 anchor 不同类的样本

目标：让 anchor 与 positive 的距离 < anchor 与 negative 的距离

In [ ]:
# TripletMarginLoss 示例
# 模拟 embedding 向量
anchor = torch.tensor([[1.0, 0.0, 0.0]])
positive = torch.tensor([[0.9, 0.1, 0.0]])   # 与 anchor 相似
negative = torch.tensor([[0.0, 1.0, 0.0]])    # 与 anchor 不同

triplet_loss = nn.TripletMarginLoss(margin=1.0)
loss = triplet_loss(anchor, positive, negative)

# 计算距离
d_ap = F.pairwise_distance(anchor, positive)
d_an = F.pairwise_distance(anchor, negative)

print(f"anchor -> positive 距离: {d_ap.item():.4f}")
print(f"anchor -> negative 距离: {d_an.item():.4f}")
print(f"TripletLoss: {loss.item():.4f}")
print(f"手动计算: max({d_ap.item():.4f} - {d_an.item():.4f} + 1.0, 0) = {max(d_ap.item() - d_an.item() + 1.0, 0):.4f}")

print("\n应用场景: 人脸识别、图像检索、推荐系统")

## 6. 损失函数总结表

PyTorch 提供了 21 种损失函数，按任务类型分类：

### 回归任务

| 损失函数 | 公式 | 适用场景 |
|---------|------|----------|
| `L1Loss` | MAE | 对异常值鲁棒 |
| `MSELoss` | MSE | 通用回归 |
| `SmoothL1Loss` | L1+MSE 结合 | 目标检测（如 Faster RCNN） |
| `HuberLoss` | 同 SmoothL1Loss | 同上，参数名不同 |
| `PoissonNLLLoss` | 泊松分布 NLL | 计数数据（事件次数预测） |
| `GaussianNLLLoss` | 高斯分布 NLL | 预测均值和方差 |

### 分类任务

| 损失函数 | 输入要求 | 适用场景 |
|---------|---------|----------|
| `CrossEntropyLoss` | 原始 logits | 多分类（最常用） |
| `NLLLoss` | log 概率 | 配合 LogSoftmax |
| `BCELoss` | 概率（0~1） | 二分类（需先 Sigmoid） |
| `BCEWithLogitsLoss` | 原始 logits | 二分类（推荐） |
| `MultiLabelMarginLoss` | - | 多标签分类 |
| `MultiLabelSoftMarginLoss` | - | 多标签分类 |
| `MultiMarginLoss` | - | 多分类 SVM |

### 排序/度量学习

| 损失函数 | 适用场景 |
|---------|----------|
| `TripletMarginLoss` | 三元组度量学习 |
| `TripletMarginWithDistanceLoss` | 自定义距离的三元组 |
| `MarginRankingLoss` | 排序学习 |
| `HingeEmbeddingLoss` | 非线性嵌入/半监督 |
| `CosineEmbeddingLoss` | 余弦相似度学习 |

### 分布匹配

| 损失函数 | 适用场景 |
|---------|----------|
| `KLDivLoss` | 知识蒸馏、VAE |
| `SoftMarginLoss` | 二分类（逻辑回归） |

### 序列任务

| 损失函数 | 适用场景 |
|---------|----------|
| `CTCLoss` | 语音识别、OCR |

## 7. 练习题

### 练习 1：在相同数据上比较不同损失函数

生成一组回归数据，分别计算 L1Loss、MSELoss、SmoothL1Loss 的值，观察差异。

In [ ]:
# 练习 1
torch.manual_seed(42)
pred = torch.randn(10)
target = torch.randn(10)

# TODO: 计算三种损失函数的值
# l1_loss = ...
# mse_loss = ...
# smooth_l1_loss = ...

# print(f"L1Loss:       {l1_loss:.4f}")
# print(f"MSELoss:      {mse_loss:.4f}")
# print(f"SmoothL1Loss: {smooth_l1_loss:.4f}")

### 练习 2：实现带 label_smoothing 的手动交叉熵

不使用 PyTorch 的 `label_smoothing` 参数，手动实现标签平滑。

In [ ]:
# 练习 2
def manual_cross_entropy_with_smoothing(logits, targets, smoothing=0.1):
    """手动实现带标签平滑的交叉熵损失。

    Args:
        logits: 原始 logits，形状 (N, C)
        targets: 类别索引，形状 (N,)
        smoothing: 标签平滑系数

    Returns:
        标量损失值
    """
    # TODO: 实现
    # 1. 构造平滑后的标签
    # 2. 计算 log_softmax
    # 3. 计算交叉熵
    pass

# 测试
# logits = torch.tensor([[2.0, 1.0, 0.1, 0.5]])
# target = torch.tensor([0])
# manual = manual_cross_entropy_with_smoothing(logits, target, smoothing=0.1)
# pytorch = nn.CrossEntropyLoss(label_smoothing=0.1)(logits, target)
# print(f"手动实现: {manual:.6f}")
# print(f"PyTorch:  {pytorch:.6f}")

### 练习 3：实现简单的知识蒸馏损失

实现一个函数，计算知识蒸馏的总损失。

In [ ]:
# 练习 3
def distillation_loss(student_logits, teacher_logits, targets, T=4.0, alpha=0.5):
    """知识蒸馏损失。

    Args:
        student_logits: 学生模型输出
        teacher_logits: 教师模型输出
        targets: 真实标签
        T: 温度参数
        alpha: 软标签权重

    Returns:
        总损失 = alpha * KL_loss + (1-alpha) * CE_loss
    """
    # TODO: 实现
    pass

# 测试
# teacher = torch.tensor([[5.0, 3.0, 1.0]])
# student = torch.tensor([[4.0, 2.5, 1.5]])
# targets = torch.tensor([0])
# loss = distillation_loss(student, teacher, targets)
# print(f"蒸馏损失: {loss:.4f}")

## 8. 小结

### 核心要点

1. **回归用 MSELoss**，对异常值鲁棒用 SmoothL1Loss
2. **多分类用 CrossEntropyLoss**（输入是原始 logits）
3. **二分类用 BCEWithLogitsLoss**（不要用 BCELoss）
4. **CrossEntropyLoss = LogSoftmax + NLLLoss**
5. `reduction` 参数控制输出形式（`'none'`, `'mean'`, `'sum'`）
6. `weight` 参数处理类别不平衡
7. `label_smoothing` 防止过拟合

### 选择指南

```
回归任务 ──> MSELoss（通用）/ SmoothL1Loss（有异常值）
二分类 ──> BCEWithLogitsLoss
多分类 ──> CrossEntropyLoss
知识蒸馏 ──> KLDivLoss
度量学习 ──> TripletMarginLoss
序列任务 ──> CTCLoss
```

### 下一步

接下来我们将学习优化器（Optimizer），了解如何用计算出的梯度来更新模型参数。